In [48]:
from reasonable_crowd.dataset import load_annotations, build_evaluation_dataset
import os
import numpy as np
path_to_reasonable_crowd = "../../Reasonable-Crowd"
path_to_workers = os.path.join(path_to_reasonable_crowd, "annotations/workers.txt")

with open(path_to_workers, "r") as f:
    workers = f.readlines()


annotations = load_annotations(path_to_reasonable_crowd)
X, y, y_votes, y_agreement = build_evaluation_dataset(annotations)

In [49]:
num_pairs = 0
for scenario, pairs in annotations.items():
    num_pairs += len(pairs)
print("Number of pairs:", num_pairs)

Number of pairs: 3364


In [76]:
worker_preferences = {}
for worker in workers:
    w_id = worker.strip()
    worker_preferences[w_id] = set()
    
for scenario, pairs in annotations.items():
    for pair_id, votes in pairs.items():
        for w_id, preferences in worker_preferences.items():
            if w_id in votes:
                t1, t2 = pair_id.split(" ;; " )
                preferences.add((t1, t2))
                

In [ ]:
agreements = []
for w_id, preferences in worker_preferences.items():
    pairs = set()
    for pref in preferences:
        pairs.add(pref)
        pairs.add(pref[::-1])
        
    for other_w_id, other_preferences in worker_preferences.items():
        if w_id == other_w_id:
            continue
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
        total_count = len(pairs.intersection(other_pairs))//2
        if total_count == 0:
            continue
        common_preferences = preferences.intersection(other_preferences)
        agreement_count = len(common_preferences)
        agreement = agreement_count / total_count if total_count > 0 else 0
        agreements.append(agreement)
agreements.sort()

agreements = np.array(agreements)

print("Annotator agreement statistics:")
print("Min agreement:", np.min(agreements))
print("Max agreement:", np.max(agreements))
print("Mean agreement:", np.mean(agreements))
print("Median agreement:", np.median(agreements))
print(len(agreements), "annotator pairs compared.")

Annotator agreement statistics:
Min agreement: 0.0
Max agreement: 1.0
Mean agreement: 0.8073593289092988
Median agreement: 0.8235887096774194
3002 annotator pairs compared.


In [92]:
total_shared_tasks = 0
total_matches = 0

for w_id, preferences in worker_preferences.items():
    pairs = set()
    for pref in preferences:
        pairs.add(pref)
        pairs.add(pref[::-1])
    
    for other_w_id, other_preferences in worker_preferences.items():
        if w_id >= other_w_id: # Avoids double-counting A-B and B-A
            continue
            
        overlap_count = len(pairs.intersection(other_pairs)) // 2
        if overlap_count == 0:
            continue
        
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
            
        matches = len(preferences.intersection(other_preferences))
        
        total_shared_tasks += overlap_count
        total_matches += matches

global_agreement = total_matches / total_shared_tasks
print(f"Global Pooled Agreement: {global_agreement:.4f}")

Global Pooled Agreement: 0.8462


In [97]:
worker_scores = {}

for w_id, preferences in worker_preferences.items():
    total_matches = 0
    total_overlap = 0
    
    # Pre-calculate the worker's own expanded pairs once
    w_pairs = set()
    for pref in preferences:
        w_pairs.add(pref)
        w_pairs.add(pref[::-1])

    for other_w_id, other_preferences in worker_preferences.items():
        if w_id == other_w_id:
            continue
            
        # Check intersection
        other_pairs = set()
        for pref in other_preferences:
            other_pairs.add(pref)
            other_pairs.add(pref[::-1])
            
        overlap = len(w_pairs.intersection(other_pairs)) // 2
        if overlap == 0:
            continue
            
        matches = len(preferences.intersection(other_preferences))
        
        total_matches += matches
        total_overlap += overlap
    
    if total_overlap > 0:
        worker_scores[w_id] = total_matches / total_overlap
    print(total_overlap)

# Statistics on workers
scores = list(worker_scores.values())
print(f"Mean Worker Reliability: {np.mean(scores)}")

1857
17715
7529
186
176
7377
6953
23924
561
23927
4517
3081
6582
88
189
9145
23399
200
5047
15264
6252
5761
973
264
13124
11572
13661
11527
19927
200
24645
23037
192
4644
11804
7007
3617
3997
12934
1258
280
120
17561
6806
19219
2895
6727
11327
12078
13979
17907
1482
16689
14366
78
17354
194
10620
14368
9123
106
1104
239
344
8123
Mean Worker Reliability: 0.8224815502621244


In [88]:
from rulebook_benchmark.rulebook import Relation

accuracies = []

for w_id, preferences in worker_preferences.items():
    correct = 0
    total = 0
    for pair, label in zip(X, y):
        t1, t2 = pair
        if (t1, t2) in preferences:
            decision = Relation.LARGER
        elif (t2, t1) in preferences:
            decision = Relation.SMALLER
        else:
            continue
    
        total += 1
        if decision == label:
            correct += 1
    accuracy = correct / total if total > 0 else 0
    accuracies.append(accuracy)
    
accuracies = np.array(accuracies)
accuracies.sort()

print("Annotator accuracy statistics:")
print("Min accuracy:", np.min(accuracies))
print("Max accuracy:", np.max(accuracies))
print("Mean accuracy:", np.mean(accuracies))
print("Median accuracy:", np.median(accuracies))
print(len(accuracies), "annotators evaluated.")
            

Annotator accuracy statistics:
Min accuracy: 0.5
Max accuracy: 1.0
Mean accuracy: 0.8363332215110086
Median accuracy: 0.8429319371727748
65 annotators evaluated.
